# TRIAGE on Kaggle T4×2 (free tier)

Full local pipeline, zero LLM API keys: generator `llama3.1:8b` + judge `qwen2.5:14b`, both via ollama.

**Before running: notebook Settings →**
- Accelerator: **GPU T4 x2**
- Internet: **ON** (requires phone-verified Kaggle account)

Session budget: **9 h per GPU commit run** (Save & Run All), ~29 GB system RAM. `prepare --rebuild` is the long pole (~75 min on 4 vCPU + T4 embedding). The benchmark checkpoints per question, so an interrupted run resumes safely. Plan: prepare + 24-question pilot in session 1; full benchmark in session 2 from restored artifacts.

**Judge options (edit `JUDGE_OLLAMA_MODEL` in cell 4):**
- `qwen2.5:14b` (default, recommended): 8b + 14b ≈ 13.7 GB — both fit one T4, no model swapping. Decent judging quality.
- `qwen2.5:7b`: faster pilot-grade fallback if VRAM gets tight.
- `qwen2.5:32b`: does NOT fit a 16 GB T4 (≈19 GB Q4); ollama CPU-offloads and judging crawls (~3–6 tok/s). Not recommended on this hardware — use the A100/Colab Pro or the `JUDGE_PROVIDER=huggingface` cloud hybrid for 32B+ quality.

In [ ]:
# 1. Clone the repo (idempotent: skips if present, pulls latest)
import sys
print(sys.version)
if sys.version_info >= (3, 13):
    print('Python 3.13 session detected - fine: cell 2 will build a Python 3.12')
    print('side-env automatically. All project cells run through {PY} - never plain python.')
!test -d TRIAGE || git clone --depth 1 https://github.com/Nirajd071/TRIAGE.git
%cd /kaggle/working/TRIAGE
!git pull --rebase 2>/dev/null; ls src/

In [ ]:
# 2. Install dependencies (10-20 min)
#    Python 3.13 session: builds a 3.12 side-env with micromamba.
#    All later cells run through {PY} - never plain python.
import sys
PY = sys.executable
if sys.version_info >= (3, 13):
    !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /usr/local bin/micromamba
    !micromamba create -y -p /kaggle/tmp/py312 python=3.12
    PY = '/kaggle/tmp/py312/bin/python'
!{PY} -m pip install -q -r requirements.txt
#    Force numpy 2.x: the resolver may pick 1.26 and the newest pyarrow
#    (via datasets/chromadb) refuses to import on numpy<2.
!{PY} -m pip install -q "numpy>=2.0,<2.6"
#    scispacy NER models with --no-deps: metadata pins spacy<3.8,
#    but they run fine under spacy 3.8 (verified on the dev machine).
!{PY} -m pip install -q --no-deps https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz
!{PY} -m pip install -q --no-deps https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_jnlpba_md-0.5.4.tar.gz
print('python:', PY)
!{PY} -c "import chromadb, spacy, scispacy; print('stack OK:', chromadb.__version__, spacy.__version__, scispacy.__version__)"

In [ ]:
# 2b. GPU verification + fix. The EMBEDDING step of prepare needs the T4;
#    the graph build is CPU by design (networkx) - CPU there is normal.
#    PyPI's default torch is a CUDA-13 build that needs a very new driver;
#    if Kaggle's driver is older, torch silently falls back to CPU. This
#    cell detects that and reinstalls the CUDA-12.6 build (~2.5 GB).
!nvidia-smi | head -12
import subprocess
r = subprocess.run([PY, '-c', 'import torch; print(torch.cuda.is_available())'], capture_output=True, text=True)
cuda_ok = r.stdout.strip().endswith('True')
print('torch.cuda.is_available():', cuda_ok)
if not cuda_ok:
    print('CUDA unavailable - reinstalling torch with CUDA 12.6 runtime...')
    !{PY} -m pip install -q --force-reinstall torch --index-url https://download.pytorch.org/whl/cu126
    r2 = subprocess.run([PY, '-c', 'import torch; print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")'], capture_output=True, text=True)
    print('after fix:', r2.stdout.strip())
else:
    !{PY} -c "import torch; print('GPU:', torch.cuda.get_device_name(0))"

In [ ]:
# 3. ollama: install (needs zstd), start server on GPU 1, pull models (~14 GB)
#    T4 x2: pin ollama to GPU 1 so GPU 0 stays free for the embedding
#    step of prepare (they collided before -> CUDA OOM at Step 5).
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import os
os.environ['OLLAMA_KEEP_ALIVE'] = '-1'   # keep both models resident in VRAM
get_ipython().system_raw('CUDA_VISIBLE_DEVICES=1 ollama serve > /tmp/ollama.log 2>&1 &')
!sleep 5 && ollama pull llama3.1:8b && ollama pull qwen2.5:14b
!ollama list

In [ ]:
# 4. Environment file - all API keys go HERE (this is the only cell with keys).
#    Preferred: Notebook editor -> Add-ons -> Secrets -> attach secrets named
#    WANDB_API_KEY and TYPESAFE_API_KEY. Secrets never appear in the notebook
#    code or its saved versions. The literals below are a fallback for
#    non-Kaggle machines only - leave them empty if you use Secrets.
#    - WANDB_API_KEY: telemetry + the off-box checkpoint backup artifact
#      (uploaded every 5 questions; the only copy that survives a killed
#      session). Without it W&B stays disabled and you rely on /kaggle/working.
#    - TYPESAFE_API_KEY: optional, live JEV Decision Gate. Without it the
#      gate runs in local calibrated mode (what the current run uses).
WANDB_API_KEY = ""      # fallback literal only
TYPESAFE_API_KEY = ""   # fallback literal only

def _secret(label):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(label)
    except Exception:
        return ""

WANDB_API_KEY = _secret("WANDB_API_KEY") or WANDB_API_KEY
TYPESAFE_API_KEY = _secret("TYPESAFE_API_KEY") or TYPESAFE_API_KEY

with open('.env', 'w') as f:
    f.write("LLM_PROVIDER=ollama\n")
    f.write("JUDGE_OLLAMA_MODEL=qwen2.5:14b\n")
    if WANDB_API_KEY:
        f.write(f"WANDB_API_KEY={WANDB_API_KEY}\n")
    else:
        f.write("WANDB_MODE=disabled\n")
    if TYPESAFE_API_KEY:
        f.write(f"TYPESAFE_API_KEY={TYPESAFE_API_KEY}\n")
# Print keys only (redacted values) so the saved notebook log stays clean.
print('\n'.join(line.split('=')[0] + '=***' for line in open('.env').read().strip().split('\n') if line.strip()))


### Optional: restore artifacts from a previous session

Skip this on a first run. After saving artifacts (bottom of this notebook) as a private Kaggle dataset, add that dataset in the notebook's right panel ("+ Add Input"), then run the cell to skip the 2–5 h `prepare --rebuild`.

In [ ]:
# 4b. RESTORE (only if you added your 'triage-artifacts' dataset as Input).
#     Handles three input styles, searched across EVERY attached Input
#     dataset (the mount path depends on the dataset slug and Kaggle's
#     current layout - e.g. /kaggle/input/datasets/<slug>/... - so nothing
#     is hardcoded):
#       1. triage_artifacts.tar.gz anywhere under /kaggle/input
#          (recommended - the archive stores data/... paths, so untarring
#          from the repo root puts everything in place),
#       2. EXTRACTED contents: a chroma.sqlite3 tree anywhere under
#          /kaggle/input - copies the vector store, graph pickle and
#          SapBERT index into place,
#       3. the legacy flat /kaggle/input/triage-artifacts layout.
#     FAILS LOUDLY with a full input listing if nothing restores: silently
#     continuing would burn the whole 9 h commit on a needless rebuild.
#     Defaults are the SESSION-2 configuration; set RESTORE=False only on
#     the very first run (no artifacts exist yet).
RESTORE = True
RESTORED = False
if RESTORE:
    import glob, os, shutil
    !mkdir -p data/processed data/vectordb
    tarballs = sorted(glob.glob('/kaggle/input/**/*.tar.gz', recursive=True)
                      + glob.glob('/kaggle/input/**/*.tgz', recursive=True))
    if tarballs:
        print(f'Restoring from {tarballs[0]} ...')
        !tar xzf "{tarballs[0]}"
    else:
        chroma_hits = glob.glob('/kaggle/input/**/chroma.sqlite3', recursive=True)
        if chroma_hits:
            src_vdb = os.path.dirname(chroma_hits[0])
            print(f'Restoring extracted store from {src_vdb} ...')
            shutil.copytree(src_vdb, 'data/vectordb', dirs_exist_ok=True)
            for fname in ('primekg_graph.gpickle', 'sapbert_node_index.faiss',
                          'sapbert_node_names.json'):
                hits = glob.glob(f'/kaggle/input/**/{fname}', recursive=True)
                if hits:
                    shutil.copy2(hits[0], f'data/processed/{fname}')
                    print(f'  + {fname}')
        else:
            !cp -r /kaggle/input/triage-artifacts/* data/processed/ 2>/dev/null || true
            !cp -r /kaggle/input/triage-artifacts/vectordb/* data/vectordb/ 2>/dev/null || true
    # Verify the restore produced a usable store before trusting it.
    chroma = 'data/vectordb/chroma.sqlite3'
    gpickle = 'data/processed/primekg_graph.gpickle'
    if os.path.exists(chroma) and os.path.getsize(chroma) > 10_000_000 \
            and os.path.exists(gpickle):
        RESTORED = True
        print(f'Restore OK: chroma.sqlite3 = {os.path.getsize(chroma)/1e6:.0f} MB')
    else:
        print('\n!! RESTORE=True but no artifacts were restored. !!')
        print('Full listing of /kaggle/input:')
        !find /kaggle/input -maxdepth 6 | head -60
        raise RuntimeError(
            'Artifact restore failed - see the /kaggle/input listing above. '
            'The attached dataset must contain triage_artifacts.tar.gz (or '
            'its extracted data/ tree). Fix the dataset, or set '
            'RESTORE=False to run a full prepare from scratch.'
        )
    !ls -la data/processed/ data/vectordb/


In [ ]:
# 5. Quick sanity: unit tests (no GPU, no network, ~5 s)
!{PY} -m pytest tests/test_api_server.py tests/test_judge_provider.py \
    tests/test_question_cache.py tests/test_pmc_downloader.py \
    tests/test_wandb_tracking.py tests/test_benchmark_metrics_wiring.py -q


In [ ]:
# 6. Build the knowledge graph + vector store (THE LONG CELL: 2-5 h).
#    SKIPPED when cell 4b restored the artifacts: prepare always
#    re-downloads PMC and re-embeds all ~794k triples (no skip path), so
#    running it after a successful restore would throw away the restored
#    store and burn ~4 h for nothing.
if RESTORED:
    print('Skipped prepare: artifacts restored in cell 4b.')
    !du -sh data/vectordb data/processed
else:
    import os
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    os.environ.setdefault('PMC_NUM_ARTICLES', '1000')
    !{PY} src/main.py prepare --rebuild


In [ ]:
# 6b. Snapshot the prepare artifacts NOW (not at the end of the notebook).
#     If the session later dies during the multi-hour benchmark, this file
#     is already in /kaggle/working, so the next session restores instead
#     of re-running the 2-5 h prepare. Re-run at the end is unnecessary:
#     the benchmark does not modify these files.
!tar czf /kaggle/working/triage_artifacts.tar.gz \
    data/processed/primekg_graph.gpickle data/processed/sapbert_node_index.faiss \
    data/processed/sapbert_node_names.json data/vectordb
!du -sh /kaggle/working/triage_artifacts.tar.gz


In [ ]:
# 7. Smoke test: one query through all three systems
!{PY} src/main.py demo

In [ ]:
# 8. Pilot benchmark (SESSION 1: ~2.5 h, 24 stratified paper-protocol
#    questions at ~6 min each). Verify results before the full run.
#    SESSION 2: set to False - the pilot already ran, and re-running it
#    would burn ~2.5 h of the 9 h GPU commit cap before the full benchmark.
RUN_PILOT = False   # True in session 1
if RUN_PILOT:
    !{PY} src/main.py base-paper --samples 24 --plot
else:
    print('Skipped pilot (RUN_PILOT=False).')


In [ ]:
# 9a. (SESSION 2, optional) Train the learned query-adaptive tensor.
#     Trains ONLY on the dev question pool - hash-disjoint from the test
#     pool the benchmark evaluates on (no data leakage). Reward is
#     NDCG@5 against each dev question's gold source chunk; no LLM judge
#     in the training loop. ~30-50 min for 60 questions x 3 epochs.
#     Skip to run the main benchmark with the static heuristic weights.
TRAIN_TENSOR = True
if TRAIN_TENSOR:
    !{PY} src/main.py train-tensor --samples 60 --epochs 3
else:
    print('Skipped tensor training (TRAIN_TENSOR=False).')


In [ ]:
# 9. (SESSION 2) Main full benchmark - STATIC weights, TEST question pool.
#    The question pool defaults to 'test' (held-out evaluation set);
#    the dev pool is reserved for training/tuning and never evaluated on.
RUN_FULL_BENCHMARK = True   # was False during session 1 (9h GPU commit cap)
if RUN_FULL_BENCHMARK:
    !{PY} src/main.py base-paper --samples 50 --plot
else:
    print('Skipped full benchmark (RUN_FULL_BENCHMARK=False).')


In [ ]:
# 9b. (SESSION 2, optional) Ablation: same cached TEST questions, LEARNED
#     tensor weights (models/dynamic_tensor.pt from cell 9a). Its own
#     checkpoint/results files so it never touches the static run's data.
#     Static-vs-learned on the identical question set = the component-level
#     ablation study the report calls for.
RUN_ABLATION = False
if RUN_ABLATION:
    import os
    os.environ['HCAD_TENSOR_MODE'] = 'learned'
    os.environ['BENCHMARK_CHECKPOINT'] = 'base_paper_checkpoint_learned.jsonl'
    os.environ['BENCHMARK_RESULTS_FILE'] = 'base_paper_benchmark_results_learned.json'
    !{PY} src/main.py base-paper --samples 50 --plot
    del os.environ['HCAD_TENSOR_MODE'], os.environ['BENCHMARK_CHECKPOINT'], os.environ['BENCHMARK_RESULTS_FILE']
else:
    print('Skipped learned-tensor ablation (RUN_ABLATION=False).')


In [ ]:
# 10. Inspect the results
import json
r = json.load(open('results/base_paper_benchmark_results.json'))
print(json.dumps({k: r[k] for k in ('scoring_based_assessment', 'retrieval_quality',
      'citation_precision', 'significance_testing')}, indent=2)[:2000])

In [ ]:
# 11. Optional: live API check (the FastAPI server from src/api/server.py)
get_ipython().system_raw(f'nohup {PY} src/main.py serve --host 127.0.0.1 --port 8000 > /tmp/serve.log 2>&1 &')
!sleep 8 && curl -s http://127.0.0.1:8000/api/triage/health
!curl -s -X POST http://127.0.0.1:8000/api/triage/query -H 'Content-Type: application/json' \
  -d '{"query":"Can Metformin be given to patients with severe renal failure given lactic acidosis risk?","config":{"tauEntropy":0.25}}' | head -c 600
!pkill -f "main.py serve"

In [ ]:
# 12. Copy the results next to the artifacts (the tarball itself was
#     already created in cell 6b right after prepare - nothing to redo).
!cp results/base_paper_checkpoint.jsonl results/base_paper_benchmark_results.json /kaggle/working/ 2>/dev/null
!cp results/base_paper_checkpoint_learned.jsonl results/base_paper_benchmark_results_learned.json /kaggle/working/ 2>/dev/null
!cp results/paper_questions_*.json /kaggle/working/ 2>/dev/null
!cp models/dynamic_tensor.pt /kaggle/working/ 2>/dev/null
!ls -la /kaggle/working/
# NOTE: /kaggle/working output limit is ~20 GB; the tarball is a few GB.
#       /kaggle/working only survives if a notebook VERSION is saved -
#       headless Save & Run All does that automatically on completion.


## Troubleshooting

- **OOM in cell 6:** Kaggle gives ~29 GB RAM; the PrimeKG graph build is the peak. If it dies, there is no swap — restart and restore from the artifacts dataset instead of re-running prepare.
- **Judge model:** default `qwen2.5:14b` fits alongside the 8b generator on one T4. `qwen2.5:7b` if tight; `qwen2.5:32b` does not fit a T4 (heavy CPU offload, very slow) — for 32B+ quality use the `JUDGE_PROVIDER=huggingface` cloud hybrid instead.
- **ollama dead after restart:** re-run cell 3 (the server doesn't survive kernel restarts).
- **Checkpoint reuse:** re-running cell 9 with the same sample count skips already-evaluated questions and aggregates over the full set.
- **9 h GPU commit cap:** prepare + pilot + full benchmark together can exceed it. That is why cell 9 is gated by RUN_FULL_BENCHMARK - do prepare + pilot in session 1, full benchmark in session 2 (restore first, cell 4b).
- **Session killed mid-benchmark:** the checkpoint (`results/base_paper_checkpoint.jsonl`) and question cache are auto-uploaded to W&B as an artifact (`benchmark-checkpoint`, every 5 questions) as long as `WANDB_API_KEY` is set. Download the latest artifact version from the `hcad-rag` W&B project, drop the two files into `results/`, restore artifacts (cell 4b), and re-run cell 9 - it resumes exactly where it died.
